# EIS Scenario Template

# Establish variables and environment

In [2]:
# import packages
import pandas as pd
import pathlib
from pathlib import Path
import os
import sys
import arcpy
import numpy as np
import pickle
# external connection packages
from sqlalchemy.engine import URL
from sqlalchemy import create_engine
# add scripts folder to path so utils and forecast_functions can be imported
sys.path.insert(0, str(pathlib.Path().absolute().parent / 'scripts'))
from utils import *
from forecast_functions import *
# pandas options
pd.options.mode.copy_on_write = True
pd.options.mode.chained_assignment = None
pd.options.display.max_columns = 999
pd.options.display.max_rows    = 999

# my workspace 
workspace = r"C:\GIS\Scratch.gdb"
# current working directory
local_path = pathlib.Path().absolute()

# get bonus_condit
# set data path as a subfolder of the current working directory 
data_dir = local_path.parents[0] / 'data'
# folder to save processed data
out_dir  = local_path.parents[0] / 'data/processed_data'
# workspace gdb for stuff that doesnt work in memory
# gdb = os.path.join(local_path,'Workspace.gdb')
gdb = workspace
# set environement workspace to in memory 
arcpy.env.workspace = 'memory'
# # clear memory workspace
# arcpy.management.Delete('memory')

# overwrite true
arcpy.env.overwriteOutput = True
# Set spatial reference to NAD 1983 UTM Zone 10N
sr = arcpy.SpatialReference(26910)

# get parcels from the database
# network path to connection files
filePath = "F:/GIS/PARCELUPDATE/Workspace/"
# database file path 
sdeBase    = os.path.join(filePath, "Vector.sde")
sdeCollect = os.path.join(filePath, "Collection.sde")
sdeTabular = os.path.join(filePath, "Tabular.sde")
sdeEdit    = os.path.join(filePath, "Edit.sde")

# Pickle variables
# part 1 - spatial joins and new categorical fields
parcel_pickle_part1    = data_dir / 'parcel_pickle1.pkl'
# part 2 - forecasting applied
parcel_pickle_part2    = data_dir / 'parcel_pickle2.pkl'

# columsn to list
initial_columns = [ 'APN',
                    'APO_ADDRESS',
                    'Residential_Units',
                    'TouristAccommodation_Units',
                    'CommercialFloorArea_SqFt',
                    'YEAR',
                    'JURISDICTION',
                    'COUNTY',
                    'OWNERSHIP_TYPE',
                    'COUNTY_LANDUSE_DESCRIPTION',
                    'EXISTING_LANDUSE',
                    'REGIONAL_LANDUSE',
                    'YEAR_BUILT',
                    'PLAN_ID',
                    'PLAN_NAME',
                    'ZONING_ID',
                    'ZONING_DESCRIPTION',
                    'TOWN_CENTER',
                    'LOCATION_TO_TOWNCENTER',
                    'TAZ',
                    'PARCEL_ACRES',
                    'PARCEL_SQFT',
                    'WITHIN_BONUSUNIT_BNDY',
                    'WITHIN_TRPA_BNDY',
                    'SHAPE']


## Establish Scenario Conditions

In [ ]:
scenario_name = 'Alternative_1'
folder_2035 = out_dir / (scenario_name + '_2035')
folder_2050 = out_dir / (scenario_name + '_2050')
# create folders if they don't exist
folder_2035.mkdir(parents=True, exist_ok=True)
folder_2050.mkdir(parents=True, exist_ok=True)

    

### Get base year parcel pickle and scenario residential zone development targets

In [ ]:
# Load base parcel data exported by parcel_engineering
#sdfParcels = pd.read_pickle(out_dir / 'base_parcel_data.pkl')
# temp hardcoded workaround
sdfParcels = pd.read_pickle(r"C:\Users\amcclary\Documents\GitHub\Transportation\TravelDemandModel\2026_Housing_EIS\Base\data\processed_data\sdf_parcel_final.pkl")
# Load lookup lists

res_zoned_lookup      = "inputs/forecast_residential_zoned_units.csv"

dfResZoned    = pd.read_csv(res_zoned_lookup)

### Establish scenario proportions

In [ ]:
# --- Pool type proportions by (Jurisdiction, Unit_Pool) ---
# Any combination not listed falls back to 'default'
zone_proportions = {
    # ('CSLT', 'Bonus Unit'): {'mf': 0.50, 'sf': 0.35, 'infill': 0.15},
    # ('TRPA', 'General'):    {'mf': 0.40, 'sf': 0.45, 'infill': 0.15},
    'default': {'mf': 0.35, 'sf': 0.50, 'infill': 0.15},
}

# --- Occupancy rates by FORECAST_REASON keyword ---
occupancy_rates = {
    'Bonus':   1.00,
    'CTC':     1.00,
    'General': 0.35,
}

# --- Household income proportions by FORECAST_REASON keyword ---
income_proportions = {
    'Bonus':   {'low': 0.78, 'medium': 0.20, 'high': 0.02},
    'General': {'low': 0.01, 'medium': 0.02, 'high': 0.97},
    'ADU':     {'low': 0.65, 'medium': 0.20, 'high': 0.15},
    'CTC':     {'low': 1.00, 'medium': 0.00, 'high': 0.00},
}

# ── Pool computation ──────────────────────────────────────────────────────────
dfPool = pd.read_csv(res_zoned_lookup)
dfPool = get_adjusted_future_units(dfPool, zone_proportions)

In [ ]:
conditions = get_parcel_conditions()
sdfParcels, df_built_parcels = forecast_jurisdiction_pools(sdfParcels, dfPool, conditions)
sdfParcels, df_built_parcels = forecast_trpa_pools(sdfParcels, dfPool, conditions, df_built_parcels)
sdfParcels, df_built_parcels =assign_remainders(sdfParcels, conditions, df_built_parcels, adu_target=4385)

In [ ]:
df_forecast_check = check_forecast_results(sdfParcels, dfPool)
sdfParcels = assign_development_year(sdfParcels, dfResZoned)
sdfParcels, dfResAssigned = assign_occupancy_rate(sdfParcels, occupancy_rates)
sdfParcels, dfIncomeAssigned = assign_income_categories(sdfParcels, income_proportions)

In [ ]:
# This should be the new base year SocioEon_Summer
socio            = 'TravelDemandModel\\2022\\data\\processed_data\\SocioEcon_Summer.csv'
socio_path       = local_path.parents[2].joinpath(socio)
dfSocio          = pd.read_csv(socio_path)

In [ ]:
dfTAZ_Summary_2035, dfTAZ_Summary_2050 = build_taz_summary(sdfParcels,dfSocio)

### Need to add code to update school enrollment and to populate employment 

### Also need code to bring in relevant files and make new folder for output file